# **Leakage Control & Train / Validation / Test Splits**
**Decodelabs Internship | Week 2**

---
This is where I created the final train / validation / test splits using a **patient-aware** strategy. Then I
applied a preprocessing pipeline (scaling, log-transforms) and verified that there is no leakage.


In [1]:
import sys, os
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from configs.config import (
    RAW_FILE, IDS_MAP_FILE, INTERIM_FILE, PROCESSED_FILE,
    TRAIN_FILE, VAL_FILE, TEST_FILE,
    FIGURES_DIR, TABLES_DIR, PAPER_FIG_DIR, PAPER_TAB_DIR,
    RANDOM_SEED, TARGET_COL, PATIENT_ID_COL, MEDICATION_COLS,
    AGE_ORDER, icd9_to_category, COLORS, ensure_dirs
)
from src.plot_utils import set_plot_style, save_figure, save_table
ensure_dirs()
set_plot_style()
print("Config loaded. Seed:", RANDOM_SEED)

Config loaded. Seed: 42


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv(PROCESSED_FILE)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique patients: {df[PATIENT_ID_COL].nunique():,}")

Loaded: 69,987 rows × 54 columns
Unique patients: 69,987


## Defining feature and target columns

I defined which columns are features (X), which is the target (y), and which is the group ID used for patient-aware splitting.

In [3]:
# Remove non-feature columns
drop_for_modelling = [TARGET_COL, PATIENT_ID_COL]

# Remove diagnosis raw category columns if they have too many unique values
# (will be one-hot encoded separately if needed — for now keep diag categories)
feature_cols = [c for c in df.columns if c not in drop_for_modelling]

print(f"Feature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:3d}. {col}")

Feature columns (52):
    1. race
    2. admission_type_id
    3. discharge_disposition_id
    4. admission_source_id
    5. time_in_hospital
    6. num_lab_procedures
    7. num_procedures
    8. num_medications
    9. number_outpatient
   10. number_emergency
   11. number_inpatient
   12. number_diagnoses
   13. metformin
   14. repaglinide
   15. nateglinide
   16. chlorpropamide
   17. glimepiride
   18. acetohexamide
   19. glipizide
   20. glyburide
   21. tolbutamide
   22. pioglitazone
   23. rosiglitazone
   24. acarbose
   25. miglitol
   26. troglitazone
   27. tolazamide
   28. examide
   29. citoglipton
   30. insulin
   31. glyburide-metformin
   32. glipizide-metformin
   33. glimepiride-pioglitazone
   34. metformin-rosiglitazone
   35. metformin-pioglitazone
   36. diag_1_cat
   37. diag_2_cat
   38. diag_3_cat
   39. age_ordinal
   40. gender_binary
   41. n_medications_active
   42. n_medication_changes
   43. n_medication_increases
   44. primary_diag_is_diabetes
 

## One-hot encode categorical string columns

The diagnosis category and race columns are still strings. I one-hot encoded them.

In [4]:
cat_string_cols = df[feature_cols].select_dtypes(include="object").columns.tolist()
print(f"String columns to one-hot encode: {cat_string_cols}")

df_encoded = pd.get_dummies(df, columns=cat_string_cols, drop_first=False, dtype=int)

# Update feature cols list after encoding
feature_cols_encoded = [c for c in df_encoded.columns if c not in drop_for_modelling]

print(f"\nShape before encoding: {df.shape}")
print(f"Shape after  encoding: {df_encoded.shape}")
print(f"New feature count: {len(feature_cols_encoded)}")

String columns to one-hot encode: ['race', 'diag_1_cat', 'diag_2_cat', 'diag_3_cat']

Shape before encoding: (69987, 54)
Shape after  encoding: (69987, 85)
New feature count: 83


## Patient-aware train / validation / test split

I used GroupShuffleSplit which splits groups (patients) entirely into one split.

In [5]:
# This guarantees: NO patient appears in both train and test.
#
# Split strategy:
#   - 70% training   (patients)
#   - 15% validation (patients)  — for hyperparameter tuning
#   - 15% test       (patients)  — final evaluation only, touch once

X = df_encoded[feature_cols_encoded]
y = df_encoded[TARGET_COL]
groups = df_encoded[PATIENT_ID_COL]   # group = patient ID

# Step A: split off 15% test set
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

X_train_val = X.iloc[train_val_idx]
y_train_val = y.iloc[train_val_idx]
groups_train_val = groups.iloc[train_val_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

# Step B: split remaining into 70% train / 15% val (val is ~15%/(70%+15%) ≈ 17.6% of train_val)
val_fraction = 0.15 / (1 - 0.15)   # ≈ 0.176
gss_val = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss_val.split(X_train_val, y_train_val, groups=groups_train_val))

X_train = X_train_val.iloc[train_idx]
y_train = y_train_val.iloc[train_idx]
groups_train = groups_train_val.iloc[train_idx]

X_val = X_train_val.iloc[val_idx]
y_val = y_train_val.iloc[val_idx]

print(f"Training set    : {len(X_train):,} rows | {y_train.mean()*100:.1f}% positive")
print(f"Validation set  : {len(X_val):,} rows  | {y_val.mean()*100:.1f}% positive")
print(f"Test set        : {len(X_test):,} rows  | {y_test.mean()*100:.1f}% positive")

Training set    : 48,990 rows | 7.5% positive
Validation set  : 10,498 rows  | 7.5% positive
Test set        : 10,499 rows  | 7.2% positive


## Verifying zero patient overlap

This is a critical leakage check. No patient should appear in more than one split.

In [6]:
train_patients = set(groups_train.values)
val_patients   = set(groups_train_val.iloc[val_idx].values)
test_patients  = set(groups.iloc[test_idx].values)

train_val_overlap  = train_patients & val_patients
train_test_overlap = train_patients & test_patients
val_test_overlap   = val_patients & test_patients

print("=== Patient Overlap Check ===")
print(f"Train ∩ Val  : {len(train_val_overlap)} patients  (should be 0)")
print(f"Train ∩ Test : {len(train_test_overlap)} patients  (should be 0)")
print(f"Val ∩ Test   : {len(val_test_overlap)} patients  (should be 0)")

if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
    print("\n✓ PASSED: No patient overlap between any splits.")
    print("This means the train/val/test split is clean and leakage-free.")
else:
    print("\n✗ FAILED: Overlap found — investigate before continuing!")

=== Patient Overlap Check ===
Train ∩ Val  : 0 patients  (should be 0)
Train ∩ Test : 0 patients  (should be 0)
Val ∩ Test   : 0 patients  (should be 0)

✓ PASSED: No patient overlap between any splits.
This means the train/val/test split is clean and leakage-free.


## Applying log1p transform to skewed features

In [7]:
# These features are right-skewed (most values are 0, a few are very large).
# log1p(x) = log(x + 1) maps 0 → 0, reduces the impact of extreme values,
# and helps linear models like Logistic Regression converge.
# Tree models don't need this, but it doesn't hurt.

skewed_cols = ["number_outpatient", "number_emergency",
               "number_inpatient", "total_prior_visits"]
skewed_cols = [c for c in skewed_cols if c in X_train.columns]

print(f"Applying log1p to {len(skewed_cols)} skewed columns: {skewed_cols}")

# Make copies to avoid modifying original DataFrames
X_train = X_train.copy()
X_val   = X_val.copy()
X_test  = X_test.copy()

for col in skewed_cols:
    X_train[col] = np.log1p(X_train[col])
    X_val[col]   = np.log1p(X_val[col])
    X_test[col]  = np.log1p(X_test[col])

print("Log1p transforms applied.")
print(f"Example: number_inpatient max before: {df[skewed_cols[2]].max()}")
print(f"         number_inpatient max after : {X_train[skewed_cols[2]].max():.2f}")

Applying log1p to 4 skewed columns: ['number_outpatient', 'number_emergency', 'number_inpatient', 'total_prior_visits']
Log1p transforms applied.
Example: number_inpatient max before: 17
         number_inpatient max after : 2.89


## Scaling features (StandardScaler)

I fitted StandardScaler ONLY on the training data to avoid leakage, then applied the same transform to validation and test data.

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)    # fit + transform on train
X_val_scaled   = scaler.transform(X_val)          # transform only on val
X_test_scaled  = scaler.transform(X_test)         # transform only on test

# Convert back to DataFrames to preserve column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=X_val.columns,   index=X_val.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_test.columns,  index=X_test.index)

print("StandardScaler applied.")
print(f"Train column means (should be ≈ 0): {X_train_scaled.mean().abs().mean():.4f}")
print(f"Train column stds  (should be ≈ 1): {X_train_scaled.std().mean():.4f}")

StandardScaler applied.
Train column means (should be ≈ 0): 0.0000
Train column stds  (should be ≈ 1): 0.9639


## Splits saved to disk

In [9]:
import os

# I attached the target back to each set for convenient saving
def make_full_df(X_scaled, y):
    d = X_scaled.copy()
    d[TARGET_COL] = y.values
    return d

train_df = make_full_df(X_train_scaled, y_train)
val_df   = make_full_df(X_val_scaled,   y_val)
test_df  = make_full_df(X_test_scaled,  y_test)

train_df.to_csv(TRAIN_FILE, index=False)
val_df.to_csv(VAL_FILE,     index=False)
test_df.to_csv(TEST_FILE,   index=False)

print(f"Saved train set : {TRAIN_FILE}  ({len(train_df):,} rows)")
print(f"Saved val set   : {VAL_FILE}   ({len(val_df):,} rows)")
print(f"Saved test set  : {TEST_FILE}  ({len(test_df):,} rows)")
print()
print("Feature columns saved:", len(feature_cols_encoded))

Saved train set : c:\Users\Peter\Documents\EXTRA-CURRICULA\Internship\Decodelab\diabetes_readmission\data\processed\train.csv  (48,990 rows)
Saved val set   : c:\Users\Peter\Documents\EXTRA-CURRICULA\Internship\Decodelab\diabetes_readmission\data\processed\val.csv   (10,498 rows)
Saved test set  : c:\Users\Peter\Documents\EXTRA-CURRICULA\Internship\Decodelab\diabetes_readmission\data\processed\test.csv  (10,499 rows)

Feature columns saved: 83
